In [1]:
# algoritmo di enumerazione
import numpy as np

def enumerate_r(A):
    n, m = A.shape
    risultati = []
    

    for i in range(n):  # riga iniziale
        for j in range(m):  # colonna iniziale
            valide = np.ones(m, dtype=bool) # inizializzo vettore valide: a priori tutte le colonne valide
            
            # (i,j) cella iniziale
            for i2 in range(i, n): # espando verso il basso fino a riga i2
                valide &= (A[i2] != 0) # se ci sono zeri nella i2-esima riga, mette False nelle rispettive colonne di valide
                
                for j2 in range(j, m): # espando verso destra fino a colonna j2
                    if valide[j2]:
                        risultati.append((i, j, i2, j2)) # 'vertici sx alto - dx basso' del rettangolo                       
                    else:
                        break 
    return risultati, len(risultati)

In [2]:
# creo la matrice per il pb di ottimizzazione
# le colonne sono i macrorettangoli, le righe le celle
# mij = 1 se macrorettangolo j può coprire cella i, 0 altrimenti

def matrice_binary(A, rect):
    # celle da coprire (solo quelle con valore 1)
    cells=np.argwhere(A==1)

    n_cells = cells.shape[0]
    n_rects = len(rect)

    M = np.zeros((n_cells, n_rects), dtype=int)

    # se gli indici i,j della cella r sono entrambi compresi rispettivamente 
    # tra gli indici i1 12 e j1 j2 del macrorettangolo k, M[r,k]=1
    for k, (i1, j1, i2, j2) in enumerate(rect):
        for r, (i, j) in enumerate(cells):
            if i1 <= i <= i2 and j1 <= j <= j2:
                M[r, k] = 1

    return M, cells

In [3]:
# creo la matrice M per il pb di ottimizzazione
# le colonne sono i macro rettangoli, le righe le celle
# mij = 1 se macro rettangolo j può coprire cella i, 0 altrimenti

def matrice_binary(A, rect):
    # celle da coprire (solo quelle con valore 1)
    cells=np.argwhere(A==1)

    n_cells = cells.shape[0]
    n_rects = len(rect)

    M = np.zeros((n_cells, n_rects), dtype=int)

    # se gli indici i,j della cella r sono entrambi compresi rispettivamente 
    # tra gli indici i1 12 e j1 j2 del macrorettangolo k, M[r,k]=1
    for k, (i1, j1, i2, j2) in enumerate(rect):
        for r, (i, j) in enumerate(cells):
            if i1 <= i <= i2 and j1 <= j <= j2:
                M[r, k] = 1

    return M, cells

In [4]:
# Esempio: istanza proposta dai proff

# definisco la matrice A che rappresenta la griglia di celle da coprire e porte/finestre
A = np.ones((4,6)) 
# metto zeri in posizione porte/finestre
A[1,1]=0
A[1,5]=0
A[3,3]=0


rect, num = enumerate_r(A)
M, cells = matrice_binary(A,rect)


# visualizzazione verticale
#for r in rect:
#   print(r)

#print("Totale rettangoli:", num)

In [5]:
# set partitioning problem

import gurobipy as gp
from gurobipy import GRB

# creo modello
m = gp.Model()

n_cells, n_rects = M.shape

# variabili
x = m.addVars(n_rects, vtype=GRB.BINARY)

# fun obiettivo
m.setObjective(gp.quicksum(x[j] for j in range(n_rects)), GRB.MINIMIZE)

# vincolo
m.addConstrs(gp.quicksum(M[i,j]*x[j] for j in range(n_rects)) == 1 for i in range(n_cells))

# risolvo
m.optimize()


Set parameter Username
Set parameter LicenseID to value 2831955
Academic license - for non-commercial use only - expires 2027-06-08
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen AI 7 350 w/ Radeon 860M, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 21 rows, 94 columns and 263 nonzeros (Min)
Model fingerprint: 0xd379fe9a
Model has 94 linear objective coefficients
Variable types: 0 continuous, 94 integer (94 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 21.0000000
Presolve removed 9 rows and 62 columns
Presolve time: 0.00s
Presolved: 12 rows, 32 columns, 86 nonzeros
Found heuristic solution: objective 14.0000000
Variable types: 0 continuous, 32 integer (32 binary)
Found heuristic s

In [6]:
# soluzioni
macrorettangoli = [j for j in range(n_rects) if x[j].x > 0.5]
print("Rettangoli selezionati:", macrorettangoli)

# quali celle copre ciascun rettangolo e totale celle coperte (per verifica)
n_celle_coperte = 0
for j in macrorettangoli:
    print("Rettangolo:", rect[j])
    celle_coperte = np.array([cells[i] for i in range(n_cells) if M[i, j] == 1])
    n_celle_coperte += len(celle_coperte)
    print("Celle coperte:",celle_coperte )

    print()
print ("totale celle coperte:", n_celle_coperte)

Rettangoli selezionati: [8, 13, 43, 70, 76, 82]
Rettangolo: (0, 0, 3, 0)
Celle coperte: [[0 0]
 [1 0]
 [2 0]
 [3 0]]

Rettangolo: (0, 1, 0, 5)
Celle coperte: [[0 1]
 [0 2]
 [0 3]
 [0 4]
 [0 5]]

Rettangolo: (1, 2, 1, 4)
Celle coperte: [[1 2]
 [1 3]
 [1 4]]

Rettangolo: (2, 1, 3, 2)
Celle coperte: [[2 1]
 [2 2]
 [3 1]
 [3 2]]

Rettangolo: (2, 3, 2, 3)
Celle coperte: [[2 3]]

Rettangolo: (2, 4, 3, 5)
Celle coperte: [[2 4]
 [2 5]
 [3 4]
 [3 5]]

totale celle coperte: 21
